# EXP10 — Validação cruzada repetida

Este notebook faz parte do pacote de reprodução. Leia as explicações antes de executar. **Candidatos não verificados permanecem sem rótulo confirmado.**

## 1. Preparar o ambiente

No Colab, a primeira célula baixa uma cópia do projeto caso ela ainda não esteja disponível. No Python local, use o ambiente e as dependências descritos no README.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def preparar_projeto():
    atual = Path.cwd()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src" / "ethmix" / "__init__.py").exists():
            raiz = pasta
            break
    else:
        raiz = Path("/content/ethereum-mixer-reproducibility")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/BorgesGusta/ethereum-mixer-reproducibility.git",
                str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))
    os.chdir(raiz)
    return raiz

RAIZ = preparar_projeto()
print("Projeto:", RAIZ)


### Bibliotecas

Esta célula instala as bibliotecas necessárias na sessão atual, caso ainda não estejam presentes.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## 2. Repetir a validação interna

Aplicamos cinco sementes e cinco divisões estratificadas. O procedimento evita reamostragem fora dos dados de treino. **Ainda não é uma avaliação independente por contrato.**

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from ethmix.features import FEATURES
from ethmix.execucao import abrir_dados

dados = abrir_dados(RAIZ / "data" / "processed" / "features_treino.parquet", ["label", *FEATURES])
X, y = dados[list(FEATURES)], dados.label.astype(int)
modelos = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "HistGradientBoosting": HistGradientBoostingClassifier(),
    "Regressão logística": LogisticRegression(max_iter=2000),
}
registros = []
for seed in [11, 22, 33, 44, 55]:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for nome, modelo in modelos.items():
        passos = [SimpleImputer(strategy="median")]
        if nome == "Regressão logística":
            passos.append(StandardScaler())
        passos.append(modelo.set_params(random_state=seed))
        pipeline = make_pipeline(*passos)
        for f1 in cross_val_score(pipeline, X, y, cv=cv, scoring="f1"):
            registros.append({"seed": seed, "modelo": nome, "f1": f1})
resultado = pd.DataFrame(registros)
display(resultado.groupby("modelo")["f1"].agg(["mean", "std"]).round(4))

## 3. Limite da demonstração

Esta célula executa uma validação repetida simples, não uma reprodução exata de todos os parâmetros do EXP-10 histórico, que registra oversampling opcional no treino e protocolos específicos. A tabela anterior não pode ser comparada diretamente ao histórico sem alinhar esses detalhes.

In [ ]:
saida = RAIZ / "results" / "generated" / "EXP10"
saida.mkdir(parents=True, exist_ok=True)
resultado.to_csv(saida / "folds_exploratorios.csv", index=False)
print("Arquivo salvo em", saida)